In [ ]:
%pip install tensorflow==2.20.0
%pip install perch-hoplite==1.0.2
%pip install -r /content/reef-pipeline/requirements.txt
# If prompted, select Runtime > Restart session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK_DIR = Path('/content/drive/MyDrive/reef_pipeline')
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import requests
import pandas as pd

manifest_url = 'https://api.figshare.com/v2/articles/29958062/versions/2'
response = requests.get(manifest_url, timeout=60)
response.raise_for_status()
files = response.json()['files']
manifest = pd.DataFrame(files)[['id', 'name', 'size', 'download_url', 'computed_md5']]
manifest_path = WORK_DIR / 'manifest.csv'
manifest.to_csv(manifest_path, index=False)
print(f'Saved {len(manifest)} manifest rows to {manifest_path}')
print('Columns:', manifest.columns.tolist())
display(manifest.head())

In [ ]:
# Run this after the project Python files are available in Colab.
# For example, put config.py, preprocessing.py, indices.py, surfperch.py,
# and pipeline.py in WORK_DIR, or change this path to their Drive folder.
import sys
sys.path.insert(0, str(WORK_DIR))

from config import PipelineConfig
from pipeline import run, collect

cfg = PipelineConfig(output_dir=WORK_DIR / 'outputs')

# Keep only the five Kenya site archives listed in the Figshare manifest.
kenya_archives = {'ken_H1.zip', 'ken_D1.zip', 'ken_N1.zip', 'ken_D2.zip', 'ken_H2.zip'}
kenya_manifest = manifest[manifest['name'].isin(kenya_archives)].copy()
missing = kenya_archives - set(kenya_manifest['name'])
if missing:
    raise ValueError(f'Expected Kenya archives missing from manifest: {sorted(missing)}')
print('Kenya archives selected:')
display(kenya_manifest[['name', 'size', 'download_url']])
run(kenya_manifest, ['clean'], cfg)
feature_path, embedding_path = collect(cfg)
print(feature_path, embedding_path)